# mmprog on Google Colab

Predicting drug-resistance-associated gene-program scores from bone marrow trephine morphology (GSE299193, Yip et al., Blood 2025).

Run order
1. Mount Drive (embeddings, crops and results persist there; raw downloads use the ephemeral disk).
2. Install the package.
3. Run the synthetic check (minutes, no download) to confirm the environment.
4. Run the smoke configuration (4 trephines), then the full cohort.

Every component prints what it does and writes its artefact under `root/runs/<run_name>/<component>/`. Re-running a cell reloads a component whose inputs are unchanged.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/mmprog'   # results, crops, embeddings
RAW = '/content/mmprog_raw'               # ephemeral: raw GEO downloads (tens of GB)
import os; os.makedirs(ROOT, exist_ok=True); os.makedirs(RAW, exist_ok=True)
print('root:', ROOT)

In [ ]:
# Get the repository. Either clone it or upload the zip next to this notebook.
import os, subprocess
if not os.path.exists('/content/mm-morphology-programs'):
    if os.path.exists('/content/mm-morphology-programs.zip'):
        subprocess.run(['unzip', '-q', '/content/mm-morphology-programs.zip', '-d', '/content'], check=True)
    else:
        subprocess.run(['git', 'clone', 'https://github.com/Abhignya-Jagathpally/mm-morphology-programs.git', '/content/mm-morphology-programs'], check=True)
%cd /content/mm-morphology-programs
!pip -q install -e ".[geo,clinical]" 2>&1 | grep -v "already satisfied" | tail -2
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. Synthetic check (no download)

In [ ]:
!python scripts/run_pipeline.py --config configs/synthetic.yaml --set root=$ROOT --set local_data=$RAW --set encoder=tiny_test --set seeds=[7] --set cv_variants=[full,shuffled] 2>&1 | tail -60

## 2. Smoke run on real data (one trephine per stage, about 20 to 30 min on a T4)

In [ ]:
!python scripts/run_pipeline.py --config configs/smoke.yaml --set root=$ROOT --set local_data=$RAW

## 3. Full cohort (all 21 trephines, 2 to 3 h on a T4; downloads dominate)

In [ ]:
!python scripts/run_pipeline.py --config configs/colab_full.yaml --set root=$ROOT --set local_data=$RAW

## 4. Read the results from the run directory, never from a screenshot

In [ ]:
import pandas as pd, json, os
run_dir = os.path.join(ROOT, 'runs', 'gse299193_full_v1')
for f in ['target_reliability.csv', 'cv_results.csv', 'cv_results_reliability_adjusted.csv', 'cv_multiseed.csv', 'program_burden.csv']:
    p = os.path.join(run_dir, f)
    if os.path.exists(p):
        print('\n==', f); display(pd.read_csv(p))
if os.path.exists(os.path.join(run_dir, 'CONTRIBUTIONS.md')):
    print(open(os.path.join(run_dir, 'CONTRIBUTIONS.md')).read())

## 5. Morphology-only inference on a new slide

In [ ]:
# !python scripts/infer_slide.py --run-dir $ROOT/runs/gse299193_full_v1 --image /content/new_slide.ome.tif --out /content/new_slide_programs.csv